# 37. 실사용 설문 155건 — 조건 개수와 검색 동작

노트북 36 이 **하드 AND 와 소프트 점수가 조건 개수에서 갈린다**는 것을 보였다.
그런데 합성 평가셋은 갈래 A(조건 4개 이상 31%)와 갈래 B(6%)의 분포가 정반대라
**어느 구간이 현실인지 알 수 없었다.**

실제 사용자가 쓴 문장으로 같은 것을 잰다.

## 0. 실행 조건과 한계

### holdout 을 건드린다는 사실을 먼저 적는다

`spec.md` §7.2 는 설문 155건을 **외부 holdout** 으로 지정했다.
`AGENTS.md` 는 *"Do not use holdout data for model or rule tuning"* 이라고 못 박는다.

**이 노트북이 재는 것은 라벨 대조가 아니다.** 정답이 없으므로 성능을 잴 수 없고,
재는 것은 **입력 분포(조건 개수)와 시스템 동작(검색 불가·후보 수·동점)** 뿐이다.

그래도 **본 사실은 남긴다.** 이후 이 155건을 라벨링해 성능을 잴 때,
*"조건 개수 분포는 이미 본 상태였다"* 를 함께 보고해야 한다.

**금지 — 이 결과를 보고 사전이나 규칙을 고치지 않는다.** 고치면 holdout 이 아니게 된다.

### 그 밖

- **API 를 호출한다.** 155건 × 1회. 프롬프트는 `15_stage1_llm_prompt_v1.txt` 그대로
- 노트북 22 가 이 중 12건을 파일럿에 썼다. **155건 전체와 남은 143건을 따로 본다**
- 조건 추출·검색 코드는 노트북 35 에서 **복사**했다. 드리프트를 막으려고
  **재현 게이트**로 노트북 35 의 `target_n` 을 다시 만들어 대조한다
- 채점(P@5)은 하지 않는다. 정답이 없다

In [1]:
import hashlib
import json
import os
import pathlib
import re
import time
import urllib.error
import urllib.request

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False
FORCE_RERUN = False

GMS_URL = "https://gms.ssafy.io/gmsapi/api.openai.com/v1/chat/completions"
MODEL = "gpt-5.4-nano"
TIMEOUT_SEC = 60
MAX_ATTEMPTS = 3
TOP_K = 5
MIN_RESULTS = 3
MAX_BIN = 6

SCHEMA_KEYS = ["scent_preference", "context", "performance",
               "avoid", "additional_requirements"]
print("REPORT_ONLY:", REPORT_ONLY, "/ FORCE_RERUN:", FORCE_RERUN)

REPORT_ONLY: False / FORCE_RERUN: False


## 1. 경로 · 해싱 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
KNOW_DIR = PROJECT_ROOT / "data" / "scent_knowledge"

INPUT_PATHS = {
    "survey": PROJECT_ROOT / "data" / "survey" / "processed"
              / "survey_nlp_queries_candidates.csv",
    "lexicon": KNOW_DIR / "domain_lexicon_v1_1.csv",
    "accord_dict": OUTPUT_DIR / "10_accord_dictionary.csv",
    "note_dict": OUTPUT_DIR / "10_note_dictionary.csv",
    "perfumes_csv": PROJECT_ROOT / "perfumes.csv",
    "prompt": OUTPUT_DIR / "15_stage1_llm_prompt_v1.txt",
    "eval_ckpt": OUTPUT_DIR / "34_evalset_stage1_checkpoint.csv",
    "eval_per_query": OUTPUT_DIR / "35_baseline_per_query.csv",
}
OUTPUT_PATHS = {
    "checkpoint": OUTPUT_DIR / "37_survey_stage1_checkpoint.csv",
    "per_query": OUTPUT_DIR / "37_survey_per_query.csv",
    "compare": OUTPUT_DIR / "37_condition_distribution.csv",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    d = hashlib.sha256()
    with pathlib.Path(path).open("rb") as h:
        for chunk in iter(lambda: h.read(1024 * 1024), b""):
            d.update(chunk)
    return d.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
PROTECTED = {p.resolve() for p in INPUT_PATHS.values()}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
survey,cb10e249489feb2e
lexicon,9e0a1365437a27a0
accord_dict,567e91370e575731
note_dict,72de5566687add52
perfumes_csv,cec1ea0b49885303
prompt,24f9c2c8f833c134
eval_ckpt,0fb2aeba10b84a1b
eval_per_query,a3a50d685986014d


## 2. 원자료 · accord 행렬 · 사전

In [3]:
survey = pd.read_csv(INPUT_PATHS["survey"])
lex = pd.read_csv(INPUT_PATHS["lexicon"], keep_default_na=False, dtype=str)
accord_dict = pd.read_csv(INPUT_PATHS["accord_dict"])
note_names = set(pd.read_csv(INPUT_PATHS["note_dict"]).iloc[:, 0].astype(str).str.lower())
perf = pd.read_csv(INPUT_PATHS["perfumes_csv"], usecols=["id", "accords"], low_memory=False)
PROMPT_TEXT = INPUT_PATHS["prompt"].read_text(encoding="utf-8")
PROMPT_SHA = hashlib.sha256(PROMPT_TEXT.encode("utf-8")).hexdigest()

print(f"설문 {len(survey)}건 / 향수 {len(perf):,} / 사전 {len(lex)}행")
print("split_status:", survey["split_status"].value_counts().to_dict()
      if "split_status" in survey.columns else "(없음)")


def parse_accords(value):
    """'name:strength|...' → [(name, strength)] 내림차순. list[tuple[str, float]]."""
    out = []
    for part in str(value).split("|"):
        if not part or part == "nan":
            continue
        n, _, s = part.partition(":")
        try:
            out.append((n, float(s)))
        except ValueError:
            continue
    out.sort(key=lambda x: (-x[1], x[0]))
    return out


pairs = perf["accords"].map(parse_accords)
ACCORDS = sorted(accord_dict["accord"])
AIDX = {a: i for i, a in enumerate(ACCORDS)}
ACC_SET = set(ACCORDS)

counts = {}
for ps in pairs:
    for n, _ in ps:
        counts[n] = counts.get(n, 0) + 1
gate = sum(counts.get(r["accord"], 0) != r["perfume_count"]
           for r in accord_dict.to_dict("records"))
print(f"[게이트 1] accord {len(ACCORDS)}개 perfume_count 오차 {gate}건")
if gate:
    raise RuntimeError("재현 게이트 실패")

M = np.zeros((len(perf), len(ACCORDS)), dtype=np.float32)
for r, ps in enumerate(pairs):
    for n, s in ps:
        j = AIDX.get(n)
        if j is not None:
            M[r, j] = s
HAS = M > 0
PID = perf["id"].to_numpy()
print(f"강도 행렬 {M.shape} / {M.nbytes/1e6:.0f}MB")

설문 155건 / 향수 131,930 / 사전 44행
split_status: {'UNASSIGNED': 155}


[게이트 1] accord 92개 perfume_count 오차 0건
강도 행렬 (131930, 92) / 49MB


## 3. 노트북 35 에서 복사한 함수

**한 글자도 바꾸지 않았다.** 다음 셀의 재현 게이트가 그것을 증명한다.

In [4]:
def norm_layers(raw):
    """향 이름 하나를 층별로 해석한다. (accord 또는 None, 층 이름)."""
    s0 = str(raw)
    if s0 in ACC_SET:
        return s0, "L0"
    s1 = re.sub(r"\s+", " ", s0.strip().lower())
    if s1 in ACC_SET:
        return s1, "L1"
    s2 = re.sub(r"s\b", "", s1).strip()
    if s2 in ACC_SET:
        return s2, "L2"
    for cand in (re.sub(r"e$", "", s2) + "y", s2 + "y"):
        if cand in ACC_SET:
            return cand, "L3"
    if s1 in note_names or s2 in note_names:
        return None, "L4_note"
    return None, "밖"


lex_acc = lex[(lex["candidate_type"] == "ACCORD") &
              (lex["target_field"] != "NO_MAPPING")].copy()
forms = {}
for r in lex.to_dict("records"):
    for f in [r["expression"]] + [a for a in r["aliases"].split("|") if a]:
        if f:
            forms.setdefault(f, set()).add(r["expression"])


def lexicon_lookup(text):
    """문장에서 사전 표현을 찾아 core accord 집합을 낸다. (set, list[str])."""
    hit_expr = set()
    for form, exprs in forms.items():
        if form and form in text:
            hit_expr |= exprs
    core = set()
    for e in hit_expr:
        g = lex_acc[lex_acc["expression"] == e]
        for r in g.to_dict("records"):
            cond = r["match_condition"]
            if cond.startswith("query_contains:"):
                toks = [t for t in cond.split(":", 1)[1].split(",") if t]
                if not any(t in text for t in toks):
                    continue
            if r["required"] == "core":
                core.add(r["candidate_name"])
    return core & ACC_SET, sorted(hit_expr)


def build_targets(parsed_json, sentence):
    """구조화 결과에서 c1/c2/c3 accord 집합과 avoid 를 만든다. dict."""
    o = json.loads(parsed_json) if isinstance(parsed_json, str) and parsed_json.strip() else {}
    scent = [str(v) for v in (o.get("scent_preference") or [])]
    add = [str(v) for v in (o.get("additional_requirements") or [])]
    avoid_raw = [str(v) for v in (o.get("avoid") or [])]
    c1 = {v for v in scent if v in ACC_SET or v.strip().lower() in ACC_SET}
    c1 = {v if v in ACC_SET else v.strip().lower() for v in c1}
    text = " ".join([str(sentence)] + add)
    lex_core, hit_expr = lexicon_lookup(text)
    c2 = c1 | lex_core
    c3 = set(c2)
    n_note, n_out = 0, 0
    for v in scent:
        acc, layer = norm_layers(v)
        if acc:
            c3.add(acc)
        elif layer == "L4_note":
            n_note += 1
        else:
            n_out += 1
    av = set()
    for v in avoid_raw:
        acc, _ = norm_layers(v)
        if acc:
            av.add(acc)
    return {"c1": sorted(c1), "c2": sorted(c2), "c3": sorted(c3),
            "avoid": sorted(av), "lex_expr": hit_expr,
            "n_note": n_note, "n_outside": n_out}


def search(acc_list, avoid_list, mode):
    """상위 TOP_K 향수의 행 인덱스. (np.ndarray, 후보 수, 5위 동점 수)."""
    cols = [AIDX[a] for a in acc_list if a in AIDX]
    if not cols:
        return np.array([], dtype=int), 0, 0
    mask = np.ones(len(perf), dtype=bool)
    for a in avoid_list:
        j = AIDX.get(a)
        if j is not None:
            mask &= ~HAS[:, j]
    if mode == "hard":
        for j in cols:
            mask &= HAS[:, j]
        score = M[:, cols].sum(axis=1)
    else:
        score = M[:, cols].mean(axis=1)
        mask &= score > 0
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return np.array([], dtype=int), 0, 0
    sc = score[idx]
    order = np.lexsort((PID[idx], -sc))
    idx = idx[order]
    tie = int((sc == sc[order][min(TOP_K, len(idx)) - 1]).sum()) if len(idx) >= TOP_K else len(idx)
    return idx[:TOP_K], int(idx.size), tie

### 재현 게이트 2 — 복사한 코드가 노트북 35 와 같은가

노트북 34 의 체크포인트 600건을 이 코드로 다시 돌려 `target_n` 을 만들고,
노트북 35 가 저장한 값과 대조한다. **하나라도 다르면 멈춘다.**

In [5]:
eck = pd.read_csv(INPUT_PATHS["eval_ckpt"])
epq = pd.read_csv(INPUT_PATHS["eval_per_query"])
ref = (epq[(epq["조건"] == "c3") & (epq["검색"] == "hard")]
       .set_index("query_id")["target_n"])
repro = {r["query_id"]: len(build_targets(r["parsed"], r["sentence"])["c3"])
         for r in eck.to_dict("records")}
diff = [q for q, v in repro.items() if int(ref.get(q, -1)) != v]
print(f"[게이트 2] 노트북 35 의 target_n {len(ref)}건 재현 → 불일치 {len(diff)}건")
if diff:
    raise RuntimeError(f"복사한 코드가 노트북 35 와 다르다: {diff[:5]}")

[게이트 2] 노트북 35 의 target_n 600건 재현 → 불일치 0건


## 4. 설문 155건 구조화 (LLM)

In [6]:
def load_key():
    """.env 에서 GMS_KEY 를 읽는다. 값을 출력하지 않는다. str."""
    key = os.environ.get("GMS_KEY", "")
    if not key:
        env = PROJECT_ROOT / ".env"
        if env.is_file():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("GMS_KEY"):
                    key = line.partition("=")[2].strip()
    if not key:
        raise RuntimeError("GMS_KEY 가 없습니다.")
    return key


def extract_json(text):
    """응답에서 JSON object 를 꺼낸다. dict 또는 None."""
    t = str(text).strip()
    if t.startswith("```"):
        t = t.split("```")[1] if "```" in t[3:] else t[3:]
        t = t[4:] if t.lower().startswith("json") else t
    i, j = t.find("{"), t.rfind("}")
    if i < 0 or j <= i:
        return None
    try:
        return json.loads(t[i:j + 1])
    except ValueError:
        return None


def schema_ok(obj):
    """spec §3 ②-a 스키마를 만족하는가. (bool, str)."""
    if not isinstance(obj, dict):
        return False, "dict 아님"
    miss = [k for k in SCHEMA_KEYS if k not in obj]
    if miss:
        return False, f"key 누락 {miss}"
    for k in ("scent_preference", "avoid", "additional_requirements"):
        if not isinstance(obj[k], list):
            return False, f"{k} 가 list 아님"
    if not isinstance(obj.get("context"), dict) or not isinstance(obj.get("performance"), dict):
        return False, "context/performance 가 dict 아님"
    return True, ""


def call_once(sentence, key):
    """LLM 1회 호출. (raw_text, usage, error)."""
    body = json.dumps({"model": MODEL,
                       "messages": [{"role": "system", "content": PROMPT_TEXT},
                                    {"role": "user", "content": str(sentence)}]}).encode("utf-8")
    req = urllib.request.Request(GMS_URL, data=body, headers={
        "Authorization": f"Bearer {key}", "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=TIMEOUT_SEC) as r:
            o = json.loads(r.read())
        return o["choices"][0]["message"]["content"], o.get("usage", {}), None
    except urllib.error.HTTPError as e:
        d = ""
        try:
            d = e.read().decode()[:200]
        except Exception:
            pass
        return None, {}, f"HTTP {e.code} {d}"
    except Exception as e:
        return None, {}, f"{type(e).__name__}: {str(e)[:160]}"

In [7]:
CKPT = OUTPUT_PATHS["checkpoint"]
done = {}
if CKPT.is_file() and not FORCE_RERUN:
    done = {r["query_id"]: r for r in pd.read_csv(CKPT).to_dict("records")}
    print(f"체크포인트에서 {len(done)}건 이어받음")

rows = survey.to_dict("records")
todo = [r for r in rows if r["query_id"] not in done]
print(f"호출 대상 {len(todo)}건 / 전체 {len(rows)}건")

if todo and not REPORT_ONLY:
    key = load_key()
    recs, t0all = [], time.time()
    for n, r in enumerate(todo, 1):
        raw, usage, err, parsed, status, why, attempts = None, {}, None, None, "ok", "", 0
        for attempt in range(1, MAX_ATTEMPTS + 1):
            attempts = attempt
            t0 = time.time()
            raw, usage, err = call_once(r["query_text"], key)
            if err and err.startswith("HTTP 401"):
                raise RuntimeError(f"GMS 키 만료로 보인다 — 멈춘다. {err}")
            if err:
                status = "api_error"
                if attempt < MAX_ATTEMPTS:
                    time.sleep(2 * attempt)
                    continue
                break
            parsed = extract_json(raw)
            if parsed is None:
                status, why = "parse_fail", "JSON 못 찾음"
                if attempt < MAX_ATTEMPTS:
                    continue
                break
            ok, why = schema_ok(parsed)
            if ok:
                status = "ok"
                break
            status = "schema_fail"
            if attempt >= MAX_ATTEMPTS:
                break
        recs.append({"query_id": r["query_id"], "query_text": r["query_text"],
                     "source": r.get("source"), "split_status": r.get("split_status"),
                     "raw_response": raw,
                     "parsed": json.dumps(parsed, ensure_ascii=False) if parsed else "",
                     "status": status, "error": err or why, "attempts": attempts,
                     "prompt_tokens": usage.get("prompt_tokens"),
                     "completion_tokens": usage.get("completion_tokens"),
                     "latency_seconds": round(time.time() - t0, 3),
                     "model_name": MODEL, "prompt_sha256": PROMPT_SHA,
                     "run_timestamp": pd.Timestamp.now("UTC").isoformat()})
        if n % 50 == 0 or n == len(todo):
            el = time.time() - t0all
            print(f"  {n}/{len(todo)}  경과 {el/60:.1f}분", flush=True)
    new = pd.DataFrame(recs)
else:
    new = pd.DataFrame()

sck = pd.concat([pd.DataFrame(list(done.values())), new], ignore_index=True) \
    if done or len(new) else pd.DataFrame()
if len(sck):
    print(f"\n체크포인트 {len(sck)}건 / 스키마 통과 {(sck['status']=='ok').mean():.1%}")
    print(f"지연 mean {sck['latency_seconds'].mean():.2f}초 / p95 "
          f"{sck['latency_seconds'].quantile(.95):.2f}초")
    # 분석이 깨져도 응답을 잃지 않도록 여기서 먼저 저장한다
    write_output(OUTPUT_PATHS["checkpoint"],
                 lambda p: sck.to_csv(p, index=False, encoding="utf-8-sig"))

호출 대상 155건 / 전체 155건


  50/155  경과 1.3분


  100/155  경과 2.6분


  150/155  경과 3.8분


  155/155  경과 4.0분



체크포인트 155건 / 스키마 통과 100.0%
지연 mean 1.54초 / p95 2.04초
저장: analysis_outputs\37_survey_stage1_checkpoint.csv


## 5. 조건 개수와 검색 동작 — 합성 평가셋과 같은 방식

In [8]:
recs = []
for r in sck.to_dict("records"):
    tg = build_targets(r["parsed"], r["query_text"])
    row = {"query_id": r["query_id"], "split_status": r.get("split_status"),
           "글자 수": len(str(r["query_text"])),
           "c1_n": len(tg["c1"]), "c2_n": len(tg["c2"]), "c3_n": len(tg["c3"]),
           "avoid_n": len(tg["avoid"]), "사전 표현": len(tg["lex_expr"])}
    for mode in ("hard", "soft"):
        _, ncand, tie = search(tg["c3"], tg["avoid"], mode)
        row[f"후보_{mode}"] = ncand
        row[f"동점_{mode}"] = tie
        row[f"검색불가_{mode}"] = int(ncand == 0)
        row[f"결과부족_{mode}"] = int(ncand < MIN_RESULTS)
    recs.append(row)
sq = pd.DataFrame(recs)
sq["n"] = sq["c3_n"].clip(upper=MAX_BIN)
print(f"설문 {len(sq)}건 처리")
display(sq[["c1_n", "c2_n", "c3_n", "avoid_n", "사전 표현"]].mean().round(2).to_frame("평균"))

설문 155건 처리


,평균
c1_n,0.56
c2_n,1.26
c3_n,1.28
avoid_n,0.14
사전 표현,0.81


## 6. 핵심 — 조건 개수 분포를 합성 평가셋과 비교

In [9]:
ev = epq[(epq["조건"] == "c3") & (epq["검색"] == "hard")].copy()
ev["n"] = ev["target_n"].clip(upper=MAX_BIN)
if "arm" not in ev.columns:
    ev = ev.merge(eck[["query_id", "arm"]], on="query_id", how="left")

dist = pd.DataFrame({
    "갈래 A": ev[ev["arm"] == "A"]["n"].value_counts(normalize=True),
    "갈래 B": ev[ev["arm"] == "B"]["n"].value_counts(normalize=True),
    "설문 155": sq["n"].value_counts(normalize=True),
}).reindex(range(MAX_BIN + 1)).fillna(0).round(3)
dist.index.name = "조건 개수"
display(dist)

print("조건 4개 이상 비율")
for c in dist.columns:
    print(f"  {c:8s} {dist.loc[4:, c].sum():.1%}")
print("\n조건 0~1개 비율")
for c in dist.columns:
    print(f"  {c:8s} {dist.loc[:1, c].sum():.1%}")

,갈래 A,갈래 B,설문 155
조건 개수,,,
0,0.117,0.333,0.484
1,0.087,0.183,0.116
2,0.263,0.263,0.200
3,0.220,0.160,0.103
4,0.170,0.040,0.052
5,0.117,0.013,0.026
6,0.027,0.007,0.019


조건 4개 이상 비율
  갈래 A     31.4%
  갈래 B     6.0%
  설문 155   9.7%

조건 0~1개 비율
  갈래 A     20.4%
  갈래 B     51.6%
  설문 155   60.0%


## 7. 설문에서 실제로 일어난 검색 실패

In [10]:
fail = sq.groupby("n").agg(**{
    "쿼리 수": ("query_id", "size"),
    "후보 중앙 하드": ("후보_hard", "median"),
    "후보 중앙 소프트": ("후보_soft", "median"),
    "동점 중앙 하드": ("동점_hard", "median"),
    "검색불가 하드": ("검색불가_hard", "mean"),
    "검색불가 소프트": ("검색불가_soft", "mean"),
    "결과부족 하드": ("결과부족_hard", "mean"),
    "결과부족 소프트": ("결과부족_soft", "mean"),
}).round(3)
display(fail)

print("전체 설문 155건 기준")
for m in ("hard", "soft"):
    print(f"  {m:5s}  검색 불가 {sq[f'검색불가_{m}'].mean():.1%} / "
          f"결과 부족 {sq[f'결과부족_{m}'].mean():.1%}")

,쿼리 수,후보 중앙 하드,후보 중앙 소프트,동점 중앙 하드,검색불가 하드,검색불가 소프트,결과부족 하드,결과부족 소프트
n,,,,,,,,
0,75,0.0,0.0,0.0,1.000,1.0,1.000,1.0
1,18,53277.5,53277.5,13991.0,0.000,0.0,0.000,0.0
2,31,3745.0,42474.0,7.0,0.065,0.0,0.065,0.0
3,16,179.0,84139.0,1.5,0.000,0.0,0.000,0.0
4,8,128.5,96655.0,1.0,0.125,0.0,0.125,0.0
5,4,30.5,85569.5,1.0,0.250,0.0,0.250,0.0
6,3,0.0,112110.0,0.0,0.667,0.0,1.000,0.0


전체 설문 155건 기준
  hard   검색 불가 52.3% / 결과 부족 52.9%
  soft   검색 불가 48.4% / 결과 부족 48.4%


## 8. 노트북 22 가 쓴 12건을 뺀 143건

`spec.md` §7.2 가 *"남은 143건"* 을 깨끗한 holdout 으로 본다. 따로 확인한다.

In [11]:
if "split_status" in sq.columns and sq["split_status"].notna().any():
    display(sq.groupby("split_status")["c3_n"].agg(["size", "mean", "median"]).round(2))
    for label, g in sq.groupby("split_status"):
        print(f"  {label}: 조건 4개 이상 {(g['c3_n']>=4).mean():.1%} / "
              f"0~1개 {(g['c3_n']<=1).mean():.1%}")
else:
    print("split_status 가 비어 있어 나눌 수 없다. 155건 전체로만 보고한다.")

,size,mean,median
split_status,,,
UNASSIGNED,155,1.28,1.0


  UNASSIGNED: 조건 4개 이상 9.7% / 0~1개 60.0%


## 9. 저장

In [12]:
if len(sck):
    write_output(OUTPUT_PATHS["per_query"],
                 lambda p: sq.to_csv(p, index=False, encoding="utf-8-sig"))
    write_output(OUTPUT_PATHS["compare"],
                 lambda p: dist.to_csv(p, encoding="utf-8-sig"))

저장: analysis_outputs\37_survey_per_query.csv
저장: analysis_outputs\37_condition_distribution.csv


## 10. 가드 검증

In [13]:
after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in after if after[k] != input_hashes_before[k]]
if changed:
    raise RuntimeError(f"입력 파일이 변경됐다: {changed}")
print("입력 해시 불변 확인:", ", ".join(INPUT_PATHS))
print()
print("주의 — 이 결과를 보고 사전이나 규칙을 고치면 설문이 holdout 이 아니게 된다.")

입력 해시 불변 확인: survey, lexicon, accord_dict, note_dict, perfumes_csv, prompt, eval_ckpt, eval_per_query

주의 — 이 결과를 보고 사전이나 규칙을 고치면 설문이 holdout 이 아니게 된다.
